In [ ]:
import os, pathlib, textwrap, glob

from langchain_community.document_loaders import UnstructuredURLLoader, TextLoader, PyPDFLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_community.embeddings import HuggingFaceEmbeddings, SentenceTransformerEmbeddings
from langchain_community.llms import Ollama
from langchain.chains import ConversationalRetrievalChain
from langchain.prompts import PromptTemplate

print("✅ Libraries imported! You're good to go!")

In [ ]:
pdf_paths = glob.glob("data/TempestTrail_*.pdf")
raw_docs = []

for path in pdf_paths:
    raw_docs.extend(PyPDFLoader(path).load())

print(f"Loaded {len(raw_docs)} PDF pages from {len(pdf_paths)} files.")

In [ ]:
URLS = [
    "https://developer.bigcommerce.com/docs/store-operations/shipping",
    "https://developer.bigcommerce.com/docs/store-operations/orders/refunds",
]

try:
    loader   = UnstructuredURLLoader(urls=URLS)    
    raw_docs = loader.load()
    print(f"Fetched {len(raw_docs)} documents from the web.")
except Exception as e:
    print("⚠️  Web fetch failed, using offline copies:", e)
    raw_docs = []
    for path in glob.glob("data/TempestTrail_*.pdf"):  
        raw_docs.extend(PyPDFLoader(path).load())
    print(f"Loaded {len(raw_docs)} offline documents.")

In [ ]:
splitter = RecursiveCharacterTextSplitter(chunk_size=300, chunk_overlap=30) 
chunks   = splitter.split_documents(raw_docs)

print(f"✅ {len(chunks)} chunks ready for embedding")

In [ ]:
embedder = SentenceTransformerEmbeddings(model_name="thenlper/gte-small")   
embedding_vector = embedder.embed_query("Hello world!")                     

print(len(embedding_vector))   

In [ ]:
vectordb  = FAISS.from_documents(chunks, embedder)          # 1. build index
retriever = vectordb.as_retriever(search_kwargs={"k": 8})   # 2. create retriever
vectordb.save_local("faiss_index")                          # 3. persist to disk

print("✅ Vector store with", vectordb.index.ntotal, "embeddings")

In [ ]:
llm      = Ollama(model="gemma3:1b", temperature=0.1)         
response = llm.invoke("Say hello in one sentence.")             
print(response)

In [ ]:
SYSTEM_TEMPLATE = """
You are a **Customer Support Chatbot**. Use only the information in CONTEXT to answer.
If the answer is not in CONTEXT, respond with "I'm not sure from the docs."

Rules:
1) Use ONLY the provided <context> to answer.
2) If the answer is not in the context, say: "I don't know based on the retrieved documents."
3) Be concise and accurate. Prefer quoting key phrases from the context.
4) When possible, cite sources as [source: source] using the metadata.

CONTEXT:
{context}

USER:
{question}
"""

In [ ]:
prompt = PromptTemplate(                                    # 1. prompt template
    input_variables=["context", "question"],
    template=SYSTEM_TEMPLATE,
)

llm = Ollama(model="gemma3:1b", temperature=0.1)           # 2. LLM

chain = ConversationalRetrievalChain.from_llm(             # 3. RAG chain
    llm=llm,
    retriever=retriever,
    combine_docs_chain_kwargs={"prompt": prompt},
    return_source_documents=True,
    verbose=False,
)


In [ ]:
test_questions = [
    "If I'm not happy with my purchase, what is your refund policy and how do I start a return?",
    "How long will delivery take for a standard order, and where can I track my package once it ships?",
    "What's the quickest way to contact your support team, and what are your operating hours?",
]

chat_history = []                                           # 1. empty history

for question in test_questions:                             # 2. loop
    result = chain({"question": question, "chat_history": chat_history})
    answer = result["answer"]
    chat_history.append((question, answer))                 # keep history growing
    print(f"\n❓ {question}")                               # 3. print
    print(f"💬 {answer}")
    print("-" * 70)
